# Nghiên cứu chất lượng RAG cho chatbot gọi món

**Mục tiêu.** Đo và cải thiện truy hồi menu, đặc biệt các yêu cầu category/tag như *Hải sản*, đồng thời giảm độ trễ và lặp phản hồi. Đây là nghiên cứu retrieval/grounding, không tuyên bố fine-tune LLM.

## 1. Câu hỏi nghiên cứu và giả thuyết

- RQ1: BM25, TF-IDF cosine và Hybrid Reciprocal-Rank Fusion (RRF) khác nhau thế nào trên cùng golden set?
- RQ2: Với yêu cầu category/tag, có nên để LLM tự chọn món không? **Giả thuyết:** deterministic live-menu grounding sẽ đạt purity=1.0 và an toàn hơn soft prompt.
- RQ3: Có thể giảm input token và thời gian phản hồi bằng candidate set ≤8, lịch sử ≤6 lượt và `reasoning_effort=low` mà vẫn giữ đúng thực đơn không?

**Tiền đăng ký lựa chọn mô hình:** dùng dev set để chọn theo MRR@5, sau đó Hit@5, sau đó P95 retrieval latency. Frozen test chỉ báo cáo cuối, không dùng để chọn. Benchmark dưới đây là chẩn đoán; không tự động thay đổi production retriever.

In [ ]:
from pathlib import Path
import sys

PROJECT_ROOT = Path.cwd().resolve().parents[1] if Path.cwd().name == 'notebooks' else Path.cwd().resolve()
sys.path.insert(0, str(PROJECT_ROOT / 'ai'))

from app.rag.knowledge_base import load_markdown_knowledge_base
from app.rag.menu_grounding import select_menu_candidates
from evaluation.retrieval_benchmark import benchmark_all, GOLDEN_CSV, KB_PATH

chunks = load_markdown_knowledge_base(KB_PATH)
len(chunks), GOLDEN_CSV

## 2. Kiểm toán dữ liệu

Nguồn tri thức markdown (FAQ/chính sách/menu mẫu) được tách khỏi **menu live** của backend. Mọi giá, availability, category và tag cho gợi ý món phải lấy từ database live tại thời điểm chat; knowledge base không được ghi đè dữ liệu này.

In [ ]:
from collections import Counter

source_distribution = Counter(chunk.source for chunk in chunks)
token_counts = [len(chunk.content.split()) for chunk in chunks]
{
    'chunks': len(chunks),
    'sources': dict(source_distribution),
    'min_words': min(token_counts),
    'max_words': max(token_counts),
    'mean_words': round(sum(token_counts) / len(token_counts), 1),
}

## 3. Ma trận phương pháp

| Phương pháp | Vai trò | Điểm mạnh | Giới hạn |
|---|---|---|---|
| BM25 + title/tag boost | lexical baseline | nhanh, giải thích được, tốt với tên món | yếu với paraphrase |
| TF-IDF cosine vector | vector baseline | đo cosine độc lập, không phụ thuộc API | không phải neural semantic embedding |
| Hybrid RRF | fusion | giảm rủi ro một ranker bỏ sót | thêm latency/độ phức tạp |
| Live category/tag grounding | hard constraint | purity và an toàn hành động | không thay thế truy hồi FAQ/chính sách |

Neural embedding chỉ được thêm vào thí nghiệm khi encoder, phiên bản, thiết bị, seed, corpus hash và chi phí được ghi nhận; không gắn nhãn sai TF-IDF là neural embedding.

In [ ]:
results = benchmark_all(top_k=5)
[result.__dict__ for result in results]

## 4. Đánh giá live-menu grounding

Metric chính cho category/tag: **candidate purity** (mọi ứng viên có đúng category/tag), **candidate coverage** (mọi món live hợp lệ có vào candidate set khi số lượng ≤8) và **action validity** (LLM chỉ được trả action ID thuộc candidate set). Đây là lớp trước LLM, vì vậy có thể unit test không cần gọi provider.

In [ ]:
menu = [
    {'id': 'sea_1', 'name': 'Nghêu hấp sả', 'category_name': 'Hải sản', 'tags': ['Hấp'], 'is_available': True},
    {'id': 'sea_2', 'name': 'Tôm rang muối', 'category_name': 'Hải sản', 'tags': ['Tôm'], 'is_available': True},
    {'id': 'main_1', 'name': 'Cơm cá kho tộ', 'category_name': 'Món chính', 'tags': ['Bữa chính'], 'is_available': True},
]
candidates = select_menu_candidates('Cho tôi các món hải sản', menu)
assert candidates and all(item['category_name'] == 'Hải sản' for item in candidates)
candidates

## 5. Quyết định triển khai và điều kiện chấp nhận

- Production giữ live category/tag grounding trước LLM; candidate set tối đa 8 món và parser action kiểm tra lại ID live.
- RAG knowledge-base tiếp tục dùng retrieval được chọn bằng protocol dev/frozen-test; benchmark này lưu số liệu và không thay bằng cảm tính.
- Prompt chỉ gửi context cần thiết, history gần nhất, memory compact; Gemini dùng `reasoning_effort=low` cho tư vấn menu ngắn.
- Regression bắt buộc: query `hải sản` không được trả món nhóm khác; tag query chỉ trả tag phù hợp; response parser loại câu lặp; action ngoài tập ứng viên bị chặn.
- P95 end-to-end phải được đo ở staging với provider thật; không được suy luận từ latency retrieval local.